# Create a local Netflix ratings CSV

Run the cells from top to bottom. The standalone generator script downloads the public Netflix Prize dataset and creates the local CSV files without loading the full dataset into memory.

The generated CSV has the columns `movie_id`, `user_id`, `rating`, and `date`.

In [14]:
import subprocess
import sys
from pathlib import Path
import numpy as np
import pandas as pd

project_path = Path.cwd()
output_path = project_path / "data" / "netflix_ratings.csv"

subprocess.run(
    [sys.executable, str(project_path / "create_netflix_ratings.py")],
    check=True,
)

Source: /Users/rayyan/.cache/kagglehub/datasets/netflix-inc/netflix-prize-data/versions/2
Output: /Users/rayyan/Code/Projects/Netflix Prize/data/netflix_ratings.csv


In [17]:
import pandas as pd

# Read a small sample to confirm the generated file is ready for EDA.
ratings_sample = pd.read_csv(
    output_path,
    nrows=100_000,
    dtype={
        "movie_id": "int32",
        "user_id": "int32",
        "rating": "int8",
    },
    parse_dates=["date"],
)
ratings_sample.head()

,movie_id,user_id,rating,date
0,1,1488844,3,2005-09-06
1,1,822109,5,2005-05-13
2,1,885013,4,2005-10-19
3,1,30878,4,2005-12-26
4,1,823519,3,2004-05-03


In [19]:
ratings = pd.read_csv('data/netflix_ratings.csv')
ratings.shape

(100480507, 4)

In [20]:
from scipy.sparse import csr_matrix, save_npz

movie_codes, movie_ids = pd.factorize(ratings["movie_id"], sort=True)
user_codes, user_ids = pd.factorize(ratings["user_id"], sort=True)

cf_csr = csr_matrix(
    (
        ratings["rating"].to_numpy(dtype=np.int8),
        (movie_codes, user_codes),
    ),
    shape=(len(movie_ids), len(user_ids)),
)

save_npz("data/netflix_ratings_csr.npz", cf_csr)
print("CSR shape:", cf_csr.shape)
print("Stored ratings:", cf_csr.nnz)

CSR shape: (17770, 480189)
Stored ratings: 100480507


In [21]:
cf_csr.shape

(17770, 480189)

In [24]:
non_nan_cells = cf_csr.nnz
total_cells = cf_csr.shape[0] * cf_csr.shape[1]
nan_cells = total_cells - non_nan_cells

print("NaN cells:", nan_cells)
print("Non-NaN cells:", non_nan_cells)
print("Total cells:", total_cells)
print("Non-Nan %",(non_nan_cells/total_cells) * 100)

NaN cells: 8432478023
Non-NaN cells: 100480507
Total cells: 8532958530
Non-Nan % 1.177557662406687


In [23]:
print("Saved: data/netflix_ratings_csr.npz")

Saved: data/netflix_ratings_csr.npz
